In [ ]:
rundir = '/gpfs/commons/home/tbotella/bam-readcount-rs/bench/results/latest'

In [ ]:
from pathlib import Path
import polars as pl
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
%config InlineBackend.figure_format = 'svg'
mpl.rcParams['figure.dpi'] = 110
mpl.rcParams['savefig.dpi'] = 200
mpl.rcParams['savefig.bbox'] = 'tight'
mpl.rcParams['font.family'] = 'DejaVu Sans'
rundir = Path(rundir)
plots = rundir / 'plots'
plots.mkdir(parents=True, exist_ok=True)
joined_dir = rundir / 'joined'
print('rundir:', rundir, '\nplots:', plots, '\nparquets:', sorted(joined_dir.iterdir())[:3])

In [ ]:
INFO = [
    'count','avg_mapping_quality','avg_basequality','avg_se_mapping_quality',
    'num_plus_strand','num_minus_strand','avg_pos_as_fraction',
    'avg_num_mismatches_as_fraction','avg_sum_mismatch_qualities',
    'num_q2_containing_reads','avg_distance_to_q2_start_in_q2_reads',
    'avg_clipped_length','avg_distance_to_effective_3p_end',
]
lf = pl.scan_parquet(str(joined_dir / '*.parquet'))
n_rows = lf.select(pl.len()).collect().item()
n_samples = sum(1 for _ in joined_dir.glob('*.parquet'))
print(f'Lazy scan over {n_samples} samples, {n_rows:,} joined rows.')

In [ ]:
# Per-feature parity metrics designed for implementation-comparison.
# Pearson r is dropped — it stays at 0.99999 even when 0.5% of values are
# wildly off, so it can't distinguish "essentially identical" from "subtly
# broken". For "does rs reproduce upstream bam-readcount?" the right framing
# is the worst-case absolute deviation across ALL rows, the fraction of rows
# that agree exactly (or at printed %.2f precision), and the Bland-Altman
# limits-of-agreement.
#
# Two passes:
#   - FULL population (all 215M+ rows): max |Δ|, %Δ=0, bias. Streaming scan.
#   - SUBSAMPLE (50/sample): %printed-eq at %.2f precision, MAE, LoA95.
#     LoA on 215M rows would be dominated by the same float-rounding noise
#     so subsample is sufficient.
import concurrent.futures

INT_METRICS = {'count', 'num_plus_strand', 'num_minus_strand', 'num_q2_containing_reads'}

def fmt_n(n):
    if n >= 1e9: return f'{n/1e9:.2f}B'
    if n >= 1e6: return f'{n/1e6:.1f}M'
    if n >= 1e3: return f'{n/1e3:.0f}k'
    return str(n)

# Pass 1 — full-population worst-case (lazy streaming over all parquets).
parquet_glob = str(joined_dir / '*.parquet')
full_stats = {}
for m in INFO:
    a, b = pl.col(f'ref_{m}'), pl.col(f'rs_{m}')
    diff = a - b
    out = (
        pl.scan_parquet(parquet_glob)
        .select(
            diff.abs().max().alias('max_abs'),
            (diff == 0).cast(pl.Float64).mean().alias('frac_zero'),
            diff.mean().alias('bias'),
            pl.len().alias('n_full'),
        )
        .collect()
    )
    full_stats[m] = {
        'n_full': int(out['n_full'][0]),
        'max_abs_dev': float(out['max_abs'][0] or 0.0),
        'pct_delta_zero': float((out['frac_zero'][0] or 0.0) * 100.0),
        'bias': float(out['bias'][0] or 0.0),
    }

# Pass 2 — uniform subsample for printed-equivalence + LoA95.
PER_FILE = 50
def _sample_one(p):
    df = pl.read_parquet(p)
    return df.sample(n=PER_FILE, seed=42) if df.height > PER_FILE else df

parquet_files = sorted(joined_dir.glob('*.parquet'))
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
    parts = list(ex.map(_sample_one, parquet_files))
sample_df = pl.concat(parts)
print(f'subsample: {sample_df.shape[0]:,} rows from {len(parquet_files)} parquets')

rows = []
for m in INFO:
    a = sample_df[f'ref_{m}'].to_numpy()
    b = sample_df[f'rs_{m}'].to_numpy()
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    is_int = m in INT_METRICS
    diff = a - b
    abs_diff = np.abs(diff)
    if is_int:
        pct_printed_eq = float((diff == 0).mean() * 100.0) if len(a) else 100.0
    else:
        # STREGA consumes %.2f-formatted strings; the operationally meaningful
        # match is whether both implementations round to the same 2 decimals.
        pct_printed_eq = float((np.round(a, 2) == np.round(b, 2)).mean() * 100.0) if len(a) else 100.0
    mae = float(abs_diff.mean()) if len(a) else 0.0
    sd = float(diff.std(ddof=1)) if len(a) > 1 else 0.0
    rows.append({
        'metric': m,
        'kind': 'int' if is_int else 'float',
        'n_full': full_stats[m]['n_full'],
        'pct_delta_zero': full_stats[m]['pct_delta_zero'],
        'pct_printed_eq': pct_printed_eq,
        'max_abs_dev': full_stats[m]['max_abs_dev'],
        'mae': mae,
        'bias': full_stats[m]['bias'],
        'loa_95': 1.96 * sd,
    })

corr_df = pl.DataFrame(rows)
corr_df.write_csv(rundir / 'per_feature_corr.tsv', separator='\t')
print(corr_df)


In [ ]:
# PER-METRIC 2D HISTOGRAM GRID — density rather than scatter (better for a
# 215M-row dataset). Title shows the new parity metrics: full-population
# max|Δ| and subsample %printed-eq.
import math
ncols = 4
nrows = math.ceil(len(INFO) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.0*ncols, 3.6*nrows))
axes = axes.flatten()
panel_letters = list('ABCDEFGHIJKLMN')
for i, m in enumerate(INFO):
    ax = axes[i]
    a = sample_df[f'ref_{m}'].to_numpy()
    b = sample_df[f'rs_{m}'].to_numpy()
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    row = corr_df.filter(pl.col('metric') == m).row(0, named=True)
    if len(a) > 0:
        ax.hist2d(a, b, bins=120, cmap='magma', cmin=1)
        lo = float(min(a.min(), b.min()))
        hi = float(max(a.max(), b.max()))
        ax.plot([lo, hi], [lo, hi], color='cyan', lw=0.8, ls='--', zorder=10)
    ax.set_xlabel(f'upstream {m}', fontsize=8)
    ax.set_ylabel(f'rs {m}', fontsize=8)
    ax.tick_params(labelsize=7)
    pe = row['pct_printed_eq']
    color = '#0a8a3a' if pe >= 99.5 else ('#c4a000' if pe >= 95.0 else '#a30000')
    title = (f'{panel_letters[i]}  {m}\n'
             f'max|Δ|={row["max_abs_dev"]:.3g}  '
             f'printed-eq={pe:.2f}%')
    ax.set_title(title, fontsize=8.5, color=color)
for j in range(len(INFO), len(axes)):
    axes[j].axis('off')
fig.suptitle(
    f'bam-readcount-rs vs upstream — 2D density per metric '
    f'({fmt_n(n_rows)} rows / {n_samples} samples, sub={fmt_n(sample_df.shape[0])})',
    fontsize=11, y=1.005,
)
fig.tight_layout()
fig.savefig(plots / 'correlation_grid.png')
plt.show()


In [ ]:
# RUNTIME / MEMORY — bam-readcount-rs vs bam-readcount Original.
# Cohorts are anonymized to cohort_1..cohort_N (deterministic, alphabetical
# on the underlying cohort name) since the underlying labels are
# access-controlled. Original-tool wall time recovered from STREGA Nextflow
# trace files.
import re
import glob

ms = pl.read_csv(rundir / 'per_sample_metrics.tsv', separator='\t').with_columns([
    pl.col('rs_wall_s').cast(pl.Float64),
    pl.col('rs_peak_rss_kb').cast(pl.Float64),
    pl.col('bed_lines').cast(pl.Int64),
])

# Anonymize cohort labels before any plotting.
real_cohorts = sorted(c for c in ms['cohort'].unique().to_list() if c is not None)
cohort_anon = {c: f'cohort_{i+1}' for i, c in enumerate(real_cohorts)}
ms = ms.with_columns(pl.col('cohort').replace_strict(cohort_anon).alias('cohort'))

def _parse_dur(s):
    if not s or s == '-':
        return None
    total = 0.0
    found = False
    for m in re.finditer(r'(\d+(?:\.\d+)?)\s*(ms|s|m|h)', s):
        v, u = float(m.group(1)), m.group(2)
        if u == 'ms': total += v / 1000.0
        elif u == 's': total += v
        elif u == 'm': total += v * 60
        elif u == 'h': total += v * 3600
        found = True
    return total if found else None

trace_root = '/gpfs/commons/groups/landau_lab/SCORPIO/PAPER_cohorts'
upstream = {}
for tf in glob.glob(f'{trace_root}/**/trace*.txt', recursive=True):
    try:
        with open(tf) as fh:
            header = fh.readline().rstrip('\n').split('\t')
            if 'name' not in header or 'realtime' not in header or 'status' not in header:
                continue
            i_name = header.index('name')
            i_status = header.index('status')
            i_real = header.index('realtime')
            for line in fh:
                parts = line.rstrip('\n').split('\t')
                if len(parts) <= max(i_name, i_status, i_real):
                    continue
                if 'BAM_READCOUNTS' not in parts[i_name]:
                    continue
                if parts[i_status] not in ('COMPLETED', 'CACHED'):
                    continue
                m = re.search(r'\(([^)]+)\)', parts[i_name])
                if not m:
                    continue
                sid = m.group(1)
                rt = _parse_dur(parts[i_real])
                if rt is None:
                    continue
                if sid not in upstream or rt > upstream[sid]:
                    upstream[sid] = rt
    except OSError:
        continue

upstream_df = pl.DataFrame({
    'sample_id': list(upstream.keys()),
    'upstream_wall_s': list(upstream.values()),
})
print(f'upstream timing recovered for {upstream_df.height}/{ms.height} samples')

ms = ms.join(upstream_df, on='sample_id', how='left')

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
cohorts_sorted = sorted(
    [c for c in ms['cohort'].unique().to_list() if c is not None],
    key=lambda c: int(c.split('_')[1]),
)
cmap = plt.get_cmap('tab10')
color_map = {c: cmap(i % 10) for i, c in enumerate(cohorts_sorted)}

ax = axes[0]
for c in cohorts_sorted:
    sub = ms.filter(pl.col('cohort') == c)
    ax.scatter(sub['bed_lines'], sub['rs_wall_s'], label=c, s=14, alpha=0.7, color=color_map[c])
ax.set_xlabel('queried sites (BED lines)')
ax.set_ylabel('wall time (s, 8 threads)')
ax.set_title(f'bam-readcount-rs runtime\nmedian {ms["rs_wall_s"].median():.1f}s across {ms.height} samples')
ax.legend(fontsize=8, frameon=False)
ax.set_xscale('log'); ax.set_yscale('log')

ax = axes[1]
ms_up = ms.filter(pl.col('upstream_wall_s').is_not_null())
for c in cohorts_sorted:
    sub = ms_up.filter(pl.col('cohort') == c)
    if sub.height == 0:
        continue
    ax.scatter(sub['bed_lines'], sub['upstream_wall_s'], label=c, s=14, alpha=0.7, color=color_map[c])
med_up = ms_up['upstream_wall_s'].median() if ms_up.height else 0.0
ax.set_xlabel('queried sites (BED lines)')
ax.set_ylabel('wall time (s)')
ax.set_title(
    f'bam-readcount Original runtime (Nextflow traces)\n'
    f'median {med_up:.0f}s across {ms_up.height} samples w/ trace'
)
ax.legend(fontsize=8, frameon=False)
ax.set_xscale('log'); ax.set_yscale('log')

ax = axes[2]
for c in cohorts_sorted:
    sub = ms.filter(pl.col('cohort') == c)
    ax.scatter(sub['bed_lines'], sub['rs_peak_rss_kb'] / 1024.0, label=c, s=14, alpha=0.7, color=color_map[c])
ax.set_xlabel('queried sites (BED lines)')
ax.set_ylabel('peak RSS (MB)')
ax.set_title(f'bam-readcount-rs memory\nmedian {ms["rs_peak_rss_kb"].median()/1024:.0f} MB')
ax.legend(fontsize=8, frameon=False)
ax.set_xscale('log')

fig.tight_layout()
fig.savefig(plots / 'runtime_memory.png')
plt.show()

both = ms.filter(pl.col('upstream_wall_s').is_not_null())
if both.height:
    speedup = (both['upstream_wall_s'] / both['rs_wall_s']).median()
    rs_med_sub = both['rs_wall_s'].median()
    print(f'on the {both.height}-sample overlap: rs median {rs_med_sub:.1f}s, upstream median {med_up:.1f}s, median speedup {speedup:.1f}x')
